### Install the libraries

In [ ]:
!pip install pinecone llama-index-vector-stores-pinecone -q

### Import the libraries

In [3]:
from llama_index.core import(
    Settings, 
    VectorStoreIndex, 
    StorageContext,
    SimpleDirectoryReader
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from llama_index.vector_stores.pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os
import time

### Initialize the APIs

In [4]:
# loading the .env file
load_dotenv()

# initializing Groq and Pinecone APIs
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
pc = Pinecone(api_key = PINECONE_API_KEY)

### LLM and Embedding Model Configuration

In [5]:
llm = Groq(model = "llama-3.1-8b-instant")
embed_model = HuggingFaceEmbedding(model_name = "sentence-transformers/all-mpnet-base-v2")

Settings.llm = llm
Settings.embed_model = embed_model

### Pinecone Vector DB Creation

In [6]:
# setting index_name
index_name = "llama3-groq-pinecone"

# checking existing indexes
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# creating an index if it doesn't exists
if index_name not in existing_indexes: 
    pc.create_index(
        name = index_name, 
        dimension = 768, 
        metric = "cosine", 
        spec = ServerlessSpec(
                cloud = "aws", 
                region = "us-east-1"
        )
    )

# connecting to the created index
index = pc.Index(index_name)
time.sleep(1)

# describing index statistics
index.describe_index_stats()

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 24}},
 'total_vector_count': 24,
 'vector_type': 'dense'}

### Loading the data

In [7]:
documents = SimpleDirectoryReader("sample_data").load_data()

### Using Pinecone Vector Store

In [8]:
vector_store = PineconeVectorStore(pinecone_index = index)
storage_context = StorageContext.from_defaults(vector_store = vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context = storage_context
)

Upserted vectors:   0%|          | 0/12 [00:00<?, ?it/s]

### Querying the stored data

In [9]:
query_engine = index.as_query_engine()
response = query_engine.query("What are the different challenges in Copilots?")

In [10]:
print(response)

While AI Copilots have the potential to revolutionize various sectors, they also come with their own set of challenges. Some of the key difficulties associated with Copilots include:

1. **Dependence on Data Quality**: The accuracy and reliability of a Copilot's output heavily rely on the quality of the data it's trained on. Poor data can lead to biased or incorrect results, which can be detrimental in critical decision-making situations.

2. **Lack of Human Judgment**: While Copilots can process vast amounts of data, they often struggle to replicate the nuance and human judgment that's essential in complex decision-making processes. This can lead to oversimplification or misinterpretation of situations.

3. **Explainability and Transparency**: As Copilots become increasingly complex, it's becoming increasingly difficult to understand how they arrive at their conclusions. This lack of transparency can make it challenging to trust their recommendations and can lead to accountability iss